In [105]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [106]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.load_clean_data import load_all_clean_data

dfs = load_all_clean_data()

from src.get_dfs import get_dfs

(circuits,
constructors,
constructor_standings,
drivers,
driver_standings,
qualifying,
races,
results) = get_dfs(dfs)

In [107]:
ml_df = results.copy()

drivers = drivers.rename(columns={
    "name": "driver_name",
    "nationality": "driver_nationality"
})

constructors = constructors.rename(columns={
    "name": "constructor_name",
    "nationality": "constructor_nationality"
})

races = races.rename(columns={
    "date": "race_date",
    "time": "race_time"
})

circuits = circuits.rename(columns={
    "locality": "circuit_locality",
    "country": "circuit_country"
})

qualifying = qualifying.rename(columns={
    "position": "qualifying_position"
})

In [108]:
ml_df.shape

(25939, 13)

In [109]:
ml_df = ml_df.merge(
    drivers,
    on="driver_id",
    how="left"
)

In [110]:
ml_df = ml_df.merge(
    constructors,
    on="constructor_id",
    how="left"
)

display(ml_df.head())
print(ml_df.shape)

,season,round,driver_id,driver_name,constructor_id,grid,position,points,laps,status,time,fastest_lap,fastest_lap_rank,given_name,family_name,dob,driver_nationality,constructor_name,constructor_nationality
0,1950,1,farina,Nino Farina,alfa,1,1,9.0,70,Finished,2:13:23.600,NaN,NaN,Nino,Farina,1906-10-30,Italian,Alfa Romeo,Swiss
1,1950,1,fagioli,Luigi Fagioli,alfa,2,2,6.0,70,Finished,+2.600,NaN,NaN,Luigi,Fagioli,1898-06-09,Italian,Alfa Romeo,Swiss
2,1950,1,reg_parnell,Reg Parnell,alfa,4,3,4.0,70,Finished,+52.000,NaN,NaN,Reg,Parnell,1911-07-02,British,Alfa Romeo,Swiss
3,1950,1,cabantous,Yves Cabantous,lago,6,4,3.0,68,+2 Laps,NaN,NaN,NaN,Yves,Cabantous,1904-10-08,French,Talbot-Lago,French
4,1950,1,rosier,Louis Rosier,lago,9,5,2.0,68,+2 Laps,NaN,NaN,NaN,Louis,Rosier,1905-11-05,French,Talbot-Lago,French


(25939, 19)


In [111]:
ml_df = ml_df.merge(
    races,
    on=["season", "round"],
    how="left"
)

In [112]:
ml_df.shape


(25939, 23)

In [113]:
ml_df = ml_df.merge(
    circuits,
    on="circuit_id",
    how="left"
)

In [114]:
ml_df.shape


(25939, 26)

In [115]:
ml_df = ml_df.merge(
    qualifying,
    on=["season", "round", "driver_id", "constructor_id"],
    how="left"
)

In [116]:
ml_df.shape

(25939, 30)

In [117]:
ml_df.head()

,season,round,driver_id,driver_name,constructor_id,grid,position,points,laps,status,...,circuit_id,race_date,race_time,circuit_name,circuit_locality,circuit_country,qualifying_position,q1,q2,q3
0,1950,1,farina,Nino Farina,alfa,1,1,9.0,70,Finished,...,silverstone,1950-05-13,NaN,Silverstone Circuit,Silverstone,UK,NaN,NaN,NaN,NaN
1,1950,1,fagioli,Luigi Fagioli,alfa,2,2,6.0,70,Finished,...,silverstone,1950-05-13,NaN,Silverstone Circuit,Silverstone,UK,NaN,NaN,NaN,NaN
2,1950,1,reg_parnell,Reg Parnell,alfa,4,3,4.0,70,Finished,...,silverstone,1950-05-13,NaN,Silverstone Circuit,Silverstone,UK,NaN,NaN,NaN,NaN
3,1950,1,cabantous,Yves Cabantous,lago,6,4,3.0,68,+2 Laps,...,silverstone,1950-05-13,NaN,Silverstone Circuit,Silverstone,UK,NaN,NaN,NaN,NaN
4,1950,1,rosier,Louis Rosier,lago,9,5,2.0,68,+2 Laps,...,silverstone,1950-05-13,NaN,Silverstone Circuit,Silverstone,UK,NaN,NaN,NaN,NaN


In [118]:
ml_df["dob"] = pd.to_datetime(ml_df["dob"])
ml_df["race_date"] = pd.to_datetime(ml_df["race_date"])

In [119]:
ml_df["driver_age"] = (
    ml_df["race_date"] - ml_df["dob"]
).dt.days / 365.25

In [120]:
ml_df["podium"] = (
    ml_df["position"] <= 3
).astype(int)

In [121]:
ml_df["win"] = (
    ml_df["position"] == 1
).astype(int)

In [122]:
ml_df["pole_position"] = (
    ml_df["grid"] == 1
).astype(int)

In [123]:
ml_df["front_row"] = (
    ml_df["grid"] <= 2
).astype(int)

In [124]:
# Points scored in the previous 5 races

points_last_5 = (
    ml_df.groupby("driver_id")["points"]
    .transform(
        lambda x: (
            x.shift(1)
            .rolling(5, min_periods=1)
            .sum()
        )
    )
)

# Add feature to dataframe

ml_df["points_last_5_races"] = points_last_5

In [125]:
# Current season driver points before current race

ml_df["current_season_points"] = (
    ml_df.groupby(["season", "driver_id"])["points"]
    .cumsum()
    - ml_df["points"]
)

In [126]:
# Total driver points per season

season_driver_points = (
    ml_df.groupby(["season", "driver_id"])["points"]
    .sum()
    .reset_index()
)

# Average points across previous seasons

season_driver_points["avg_points_per_season"] = (
    season_driver_points
    .groupby("driver_id")["points"]
    .transform(
        lambda x: (
            x.shift(1)
            .expanding()
            .mean()
        )
    )
)

# Merge back

ml_df = ml_df.merge(
    season_driver_points[
        ["season", "driver_id", "avg_points_per_season"]
    ],
    on=["season", "driver_id"],
    how="left"
)

In [127]:
# Constructor points scored per race

constructor_round_points = (
    ml_df.groupby(
        ["season", "round", "constructor_id"]
    )["points"]
    .sum()
    .reset_index()
)

# Cumulative constructor points BEFORE current race

constructor_round_points[
    "constructor_points_current_season"
] = (
    constructor_round_points
    .groupby(["season", "constructor_id"])["points"]
    .cumsum()
    - constructor_round_points["points"]
)

# Merge back to main dataframe

ml_df = ml_df.merge(
    constructor_round_points[
        [
            "season",
            "round",
            "constructor_id",
            "constructor_points_current_season"
        ]
    ],
    on=["season", "round", "constructor_id"],
    how="left"
)

In [128]:
ml_df.shape

(25939, 39)

In [129]:
ml_df["grid"] = pd.to_numeric(
    ml_df["grid"],
    errors="coerce"
).astype("Int64")

ml_df["qualifying_position"] = pd.to_numeric(
    ml_df["qualifying_position"],
    errors="coerce"
).astype("Int64")

In [132]:
categorical_columns = [
    "driver_id",
    "driver_name",
    "constructor_id",
    "status",
    "given_name",
    "family_name",
    "driver_nationality",
    "constructor_name",
    "constructor_nationality",
    "race_name",
    "circuit_id",
    "circuit_name",
    "circuit_locality",
    "circuit_country"
]

for col in categorical_columns:
    ml_df[col] = ml_df[col].astype("category")


ml_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25939 entries, 0 to 25938
Data columns (total 39 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   season                             25939 non-null  int64         
 1   round                              25939 non-null  int64         
 2   driver_id                          25939 non-null  category      
 3   driver_name                        25939 non-null  category      
 4   constructor_id                     25939 non-null  category      
 5   grid                               25939 non-null  Int64         
 6   position                           25939 non-null  int64         
 7   points                             25939 non-null  float64       
 8   laps                               25939 non-null  int64         
 9   status                             25939 non-null  category      
 10  time                               8354 non-n

In [131]:
ml_df.to_csv(
    "../data/processed/master_ml_dataset.csv",
    index=False
)
